# Trois lectures du même panier · *Three readings of the same basket*

Notebook compagnon de l'enquête **L'or monte-t-il parce que les monnaies s'effondrent ?** — [lire l'article](https://nmlab.io/ressources/prix-de-l-or-et-effondrement-des-monnaies).
Companion notebook to the study **Is gold rising because currencies are collapsing?**.

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données publiques du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's public data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import io
import re
import urllib.request
from functools import lru_cache

import numpy as np
import pandas as pd
from pandas import DataFrame, Series

CMO_PAGE = "https://www.worldbank.org/en/research/commodity-markets"
CMO_FILE = ("https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026"
            "/related/CMO-Historical-Data-Monthly.xlsx")
FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"

# H.10 : EUR, GBP et AUD sont cotés en dollars par unité étrangère — on les inverse
# pour obtenir partout des unités locales par dollar, comme dans l'article.
# H.10 quotes EUR, GBP and AUD as dollars per foreign unit: invert them so every
# series is local units per dollar, as in the article.
FX = {"EUR": ("DEXUSEU", True), "JPY": ("DEXJPUS", False), "GBP": ("DEXUSUK", True),
      "CHF": ("DEXSZUS", False), "CAD": ("DEXCAUS", False), "AUD": ("DEXUSAL", True),
      "CNY": ("DEXCHUS", False)}


def _fetch(url: str, tries: int = 4) -> bytes:
    """Télécharge une URL, avec quelques reprises (FRED coupe parfois la connexion).
    Download a URL, retrying a few times (FRED occasionally drops the connection)."""
    import time
    for attempt in range(tries):
        try:
            return urllib.request.urlopen(url, timeout=90).read()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("unreachable")


@lru_cache(maxsize=None)
def load_gold_usd() -> Series:
    """Or en dollars par once, moyennes mensuelles depuis 1960.

    Source : « Commodity Price Data » (Pink Sheet) de la Banque mondiale, feuille
    « Monthly Prices », colonne Gold — le fixing de Londres, en accès libre.
    World Bank Pink Sheet, monthly London gold price in US dollars per troy ounce.
    """
    try:
        raw = _fetch(CMO_FILE)
    except Exception:                                  # millésime renouvelé : on relit le lien
        page = _fetch(CMO_PAGE).decode("utf-8", "ignore")
        link = re.search(r"https://[^\"']*CMO-Historical-Data-Monthly\.xlsx", page)
        raw = _fetch(link.group(0))
    table = pd.read_excel(io.BytesIO(raw), sheet_name="Monthly Prices", skiprows=4)
    table = table.rename(columns={table.columns[0]: "date"})[["date", "Gold"]].dropna()
    dates = pd.to_datetime(table["date"].str.replace("M", "-"), format="%Y-%m")
    return Series(table["Gold"].values, index=dates).astype(float)


@lru_cache(maxsize=None)
def load_fred(series_id: str) -> Series:
    """Série FRED (CSV public, sans clé) ramenée à des moyennes mensuelles.
    A FRED series (public CSV, no key) averaged to monthly frequency."""
    table = pd.read_csv(io.StringIO(_fetch(FRED_CSV.format(series_id)).decode()))
    values = pd.to_numeric(table[table.columns[1]], errors="coerce")
    series = Series(values.values, index=pd.to_datetime(table[table.columns[0]])).dropna()
    return series.resample("MS").mean()


def load_gold_in_currencies(start: str, end: str) -> DataFrame:
    """Prix de l'or dans les huit devises du panier, mois par mois.

    Chaque prix local est le produit de la moyenne mensuelle de l'or en dollars
    et de la moyenne mensuelle du taux de change — l'ordre des opérations retenu
    par l'article. Le dollar vaut 1 par construction.
    Gold priced in the eight basket currencies, month by month.
    """
    gold = load_gold_usd()
    prices = {"USD": gold}
    for code, (series_id, invert) in FX.items():
        rate = load_fred(series_id)
        prices[code] = gold * (1 / rate if invert else rate)
    return DataFrame(prices).loc[start:end].dropna()


def effective_index(prices: DataFrame) -> Series:
    """Indice or effectif : moyenne géométrique équipondérée des huit prix locaux,
    base 100 au premier mois. Seule la moyenne géométrique garantit que l'indice
    des devises mesurées contre l'or est exactement l'inverse de celui-ci.
    Equal-weighted geometric mean of the eight local prices, first month = 100.
    """
    return 100 * np.exp(np.log(prices / prices.iloc[0]).mean(axis=1))


import json

ZONES = ["US", "EA", "JP", "UK", "CH", "CA", "AU", "CN"]
Q0, QT = "1999Q1", "2026Q1"                # fenêtre commune imposée par l'IPC australien

OECD = ("https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,{dsd}@{flow},1.0/"
        "{key}?startPeriod=1998-12&format=csvfile")
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/126 Safari/537.36"}


def _fetch_h(url: str, headers: dict | None = None, data: str | None = None, tries: int = 4) -> bytes:
    """Comme _fetch, mais avec en-têtes et corps POST — certains diffuseurs les exigent.
    Same as _fetch, with headers and POST body: some publishers require them."""
    import time
    for attempt in range(tries):
        try:
            request = urllib.request.Request(url, headers=headers or {},
                                             data=data.encode() if data else None)
            return urllib.request.urlopen(request, timeout=120).read()
        except Exception:
            if attempt == tries - 1:
                raise
            time.sleep(3 * (attempt + 1))
    raise RuntimeError("unreachable")


def _monthly(index, values) -> Series:
    return Series(np.asarray(values, dtype=float), index=pd.PeriodIndex(index, freq="M")).sort_index()


def _fred_periods(series_id: str) -> Series:
    series = load_fred(series_id)
    return _monthly(series.index.to_period("M"), series.values)


def _oecd(key: str, coicop2018: bool = False) -> Series:
    """Indice de prix à la consommation diffusé par l'OCDE (SDMX public, format CSV).
    Consumer price index from the OECD SDMX public endpoint."""
    dsd, flow = (("DSD_PRICES_COICOP2018", "DF_PRICES_C2018_ALL") if coicop2018
                 else ("DSD_PRICES", "DF_PRICES_ALL"))
    # L'OCDE refuse les clients sans en-tête de navigateur (403) ; FRED et la RBA
    # refusent l'inverse. D'où deux fonctions de téléchargement distinctes.
    # The OECD rejects clients without a browser header (403); FRED and the RBA reject
    # the opposite. Hence the two separate download helpers.
    table = pd.read_csv(io.StringIO(_fetch_h(OECD.format(dsd=dsd, flow=flow, key=key), UA).decode()))
    time_col = [c for c in table.columns if "TIME" in c][0]
    value_col = [c for c in table.columns if c.startswith("OBS_VALUE")][0]
    table = table[[time_col, value_col]].dropna().sort_values(time_col)
    freq = "Q" if "Q" in str(table[time_col].iloc[0]) else "M"
    return Series(table[value_col].values.astype(float),
                  index=pd.PeriodIndex(table[time_col], freq=freq)).sort_index()


def load_cpi() -> dict[str, Series]:
    """Les huit indices de prix, chacun à sa source nationale ou à l'OCDE.
    The eight consumer price indices, each from its national source or the OECD."""
    return {
        "US": _fred_periods("CPIAUCNS"),                        # BLS, non désaisonnalisé
        "EA": _fred_periods("CP0000EZ19M086NEST"),              # IPCH, Eurostat
        "JP": _oecd("JPN.M.N.CPI.IX._T.N._Z", True),
        "UK": _oecd("GBR.M.N.CPI.IX._T.N._Z"),
        "CH": _oecd("CHE.M.N.CPI.IX._T.N._Z", True),
        "CA": _oecd("CAN.M.N.CPI.IX._T.N._Z", True),
        "AU": _oecd("AUS.Q.N.CPI.IX._T.N._Z"),                  # trimestriel à la source
        "CN": _oecd("CHN.M.N.CPI.IX._T.N._Z"),
    }


def _ecb_m3() -> Series:
    """M3 de la zone euro (BCE, base de données des statistiques monétaires BSI)."""
    url = ("https://data-api.ecb.europa.eu/service/data/BSI/"
           "M.U2.Y.V.M30.X.I.U2.2300.Z01.E?format=csvdata")
    table = pd.read_csv(io.StringIO(_fetch(url).decode()))
    return _monthly(pd.PeriodIndex(table["TIME_PERIOD"], freq="M"), table["OBS_VALUE"])


def _boe_m4() -> Series:
    """M4 du Royaume-Uni (Banque d'Angleterre, série LPMAUYN)."""
    url = ("https://www.bankofengland.co.uk/boeapps/database/_iadb-fromshowcolumns.asp?csv.x=yes"
           "&Datefrom=01/Jan/1998&Dateto=01/Dec/2026&SeriesCodes=LPMAUYN&CSVF=TN"
           "&UsingCodes=Y&VPD=Y&VFD=N")
    table = pd.read_csv(io.StringIO(_fetch_h(url, UA).decode()))
    return _monthly(pd.to_datetime(table["DATE"], format="%d %b %Y").dt.to_period("M"),
                    table["LPMAUYN"])


def _snb_m3() -> Series:
    """M3 suisse (BNS, cube snbmonagg : dimension B = niveau, GM3 = agrégat M3)."""
    raw = _fetch("https://data.snb.ch/api/cube/snbmonagg/data/csv/en").decode("utf-8-sig")
    rows = [line.split(";") for line in raw.splitlines() if line.count(";") == 3]
    table = pd.DataFrame(rows[1:], columns=[c.strip('"') for c in rows[0]]).map(lambda x: x.strip('"'))
    table = table[(table["D0"] == "B") & (table["D1"] == "GM3")]
    return _monthly(pd.PeriodIndex(table["Date"], freq="M"),
                    pd.to_numeric(table["Value"], errors="coerce"))


def _boc_m2pp() -> Series:
    """M2++ canadien (Banque du Canada, série V41552801 via l'API Valet)."""
    raw = _fetch("https://www.bankofcanada.ca/valet/observations/V41552801/csv"
                 "?start_date=1998-01-01").decode()
    body = raw.split('"OBSERVATIONS"')[1].strip().splitlines()
    table = pd.DataFrame([r.replace('"', "").split(",") for r in body[1:]], columns=["date", "value"])
    return _monthly(pd.PeriodIndex(table["date"], freq="M"),
                    pd.to_numeric(table["value"], errors="coerce"))


def _rba_broad() -> Series:
    """Monnaie large australienne (RBA, tableau D3 des agrégats monétaires)."""
    raw = _fetch("https://www.rba.gov.au/statistics/tables/csv/d3-data.csv").decode("utf-8-sig")
    lines = raw.splitlines()
    head = next(i for i, line in enumerate(lines) if line.startswith("Series ID"))
    header = [c.strip('"') for c in lines[head].split(",")]
    column = next(i for i, code in enumerate(header) if code.startswith("DMABM"))
    rows = [line.split(",") for line in lines[head + 1:] if line and line[0].isdigit()]
    table = pd.DataFrame(rows)
    return _monthly(pd.PeriodIndex(pd.to_datetime(table[0], dayfirst=True), freq="M"),
                    pd.to_numeric(table[column], errors="coerce")).dropna()


def _china_m2() -> Series:
    """M2 chinoise de fin de mois : diffusion MOFCOM (module 047), complétée par le
    pont FMI/FRED pour 1999 et par le Bureau national des statistiques pour 2026.
    Chinese end-of-month M2: MOFCOM module 047, bridged with IMF/FRED and NBS."""
    raw = _fetch_h("https://data.mofcom.gov.cn/datamofcom/front/zhtj/dateQuery",
                   {**UA, "X-Requested-With": "XMLHttpRequest",
                    "Content-Type": "application/x-www-form-urlencoded",
                    "Referer": "https://data.mofcom.gov.cn/zhtj/coin.shtml"},
                   "start_date=1999-01&end_date=2026-12&type=7&module_no=047")
    rows = json.loads(raw.decode())[0]
    money = _monthly([f"{r['time_view'][:4]}-{r['time_view'][4:]}" for r in rows],
                     [r["value"] for r in rows])                      # en 亿元 (10⁸ yuans)
    bridge = _fred_periods("MYAGM2CNM189N") / 1e8                     # yuans → 亿元
    latest = json.loads(_fetch("https://api.db.nomics.world/v22/series/NBS/M_A0D01/A0D0101"
                               "?observations=1").decode())["series"]["docs"][0]
    for extra in (bridge, _monthly(latest["period"], latest["value"])):
        money = money.combine_first(extra)
    return money.sort_index()


def _japan_m3() -> Series:
    """M3 japonaise : niveaux OCDE/FRED, prolongés par les variations annuelles de la BOJ.

    La Banque du Japon ne publie plus les niveaux dans un format lisible sans clé ; ses
    variations sur douze mois (tableau MD02) permettent de prolonger la série de niveaux.
    Japanese M3: OECD/FRED levels extended with the BOJ's year-on-year changes.
    """
    levels = _fred_periods("MABMM301JPM189S")
    try:
        page = _fetch("https://www.stat-search.boj.or.jp/ssi/mtshtml/md02_m_1.html")
        table = pd.read_html(io.StringIO(page.decode("shift_jis", "ignore")))[0]
        rows = table.iloc[5:, [0, 2]].dropna()                        # date, M3 en % sur un an
        rows = rows[rows[0].astype(str).str.match(r"\d{4}/\d{2}")]
        growth = _monthly(pd.PeriodIndex(rows[0].astype(str).str.replace("/", "-"), freq="M"),
                          pd.to_numeric(rows[2], errors="coerce"))
        for period in growth.index:
            if period in levels.index or period - 12 not in levels.index or pd.isna(growth[period]):
                continue
            levels.loc[period] = levels.loc[period - 12] * (1 + growth[period] / 100)
    except Exception as error:                                        # la série reste utilisable
        print(f"[or] prolongation BOJ indisponible ({error})")
    return levels.sort_index()


def load_money() -> dict[str, Series]:
    """Les huit agrégats de monnaie large, chacun à sa banque centrale.
    The eight broad money aggregates, each from its own central bank."""
    return {"US": _fred_periods("M2SL"), "EA": _ecb_m3(), "JP": _japan_m3(), "UK": _boe_m4(),
            "CH": _snb_m3(), "CA": _boc_m2pp(), "AU": _rba_broad(), "CN": _china_m2()}


def to_quarterly(series: Series) -> Series:
    """Moyenne des trois mois du trimestre ; les séries déjà trimestrielles sont gardées telles quelles.
    Quarterly average; natively quarterly series are kept as published."""
    if series.index.freqstr.startswith("Q"):
        return series
    return series.groupby(series.index.asfreq("Q")).mean()


def basket(series_by_zone: dict[str, Series]) -> Series:
    """Indice composite : moyenne géométrique équipondérée des huit séries, base 100 en 1999 T1.
    Equal-weighted geometric mean of the eight series, 1999 Q1 = 100."""
    frame = pd.DataFrame({zone: to_quarterly(series) for zone, series in series_by_zone.items()})
    frame = frame.loc[Q0:QT]
    return 100 * np.exp(np.log(frame / frame.loc[Q0]).mean(axis=1))


def gold_basket() -> Series:
    """Indice or effectif en trimestriel, construit mois par mois puis moyenné.
    Effective gold index, built monthly then averaged to quarters."""
    prices = load_gold_in_currencies("1999-01-01", "2026-03-01")
    prices.index = prices.index.to_period("M")
    return basket({zone: prices[zone] for zone in prices.columns})


from matplotlib.figure import Figure
from matplotlib.ticker import FixedLocator, FuncFormatter

LABELS = {
    "fr": dict(
        title="Ce qui reste de la hausse, dénominateur après dénominateur",
        sub="Indice or effectif du panier de huit zones, base 100 au premier trimestre 1999 — échelle logarithmique.",
        raw="Indice or effectif", cpi="Après les huit IPC locaux",
        money="Après les huit agrégats de monnaie large",
        note="Diviser n'est pas expliquer : le rapport dit ce qui subsiste relativement à chaque dénominateur,\n"
             "pas ce qui a causé la hausse. Sources : Banque mondiale, Réserve fédérale, OCDE, BCE, BoE, BNS, BoC, RBA, MOFCOM."),
    "en": dict(
        title="What is left of the rise, denominator after denominator",
        sub="Effective gold index for the eight-zone basket, 1999 Q1 = 100 — logarithmic scale.",
        raw="Effective gold index", cpi="After the eight local CPIs",
        money="After the eight broad money aggregates",
        note="Dividing is not explaining: the ratio says what survives relative to each denominator, not what caused\n"
             "the rise. Sources: World Bank, Federal Reserve, OECD, ECB, BoE, SNB, BoC, RBA, MOFCOM."),
}


def build_figure(gold: Series, cpi: Series, money: Series, lang: str) -> Figure:
    """Les trois courbes du panier : brut, déflaté par les prix, rapporté à la monnaie large."""
    text = LABELS[lang]
    curves = ((gold, "raw", nm.COLORS["text"], 3.6),
              (100 * gold / cpi, "cpi", nm.COLORS["teal"], 3.0),
              (100 * gold / money, "money", nm.COLORS["amber"], 3.0))

    fig = nm.figure(height_px=1120)
    ax = nm.axes(fig, left=0.075, right=0.982)
    for series, key, color, width in curves:
        # Le facteur final va dans la légende : en fin de période les trois courbes se
        # rapprochent, et des étiquettes posées sur les points se chevaucheraient.
        factor = f"×{series.iloc[-1] / 100:.2f}".replace(".", "," if lang == "fr" else ".")
        ax.plot(series.index.to_timestamp(how="end"), series.values, color=color, lw=width,
                label=f"{text[key]}  ·  {factor}", zorder=4)

    ax.set_yscale("log")
    ax.yaxis.set_major_locator(FixedLocator([50, 100, 200, 500, 1000, 2000]))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.yaxis.set_major_formatter(FuncFormatter(
        lambda v, _: f"{v:,.0f}".replace(",", " " if lang == "fr" else ",")))
    legend = ax.legend(loc="upper left", frameon=False, fontsize=20.5, labelcolor="linecolor",
                       handlelength=1.6, borderaxespad=1.2)
    for handle in legend.get_lines():
        handle.set_linewidth(3.4)

    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig


build_figure(gold_basket(), basket(load_cpi()), basket(load_money()), LANG)